In [1]:
import torch
from torch.utils.data import DataLoader, random_split, Dataset
from torchmetrics import F1Score, AUROC, JaccardIndex
from tqdm import tqdm
import cv2
import numpy as np
from PIL import Image
import os
import albumentations as A
from segmentation_models_pytorch import Unet
import copy
# Custom
from UNet import UNet

In [2]:
# Load the data
base_path = "train/"
images = []
masks = []
labels = []

# have to resize all images to the same size, it does speed things up in the future but mostly because some of the data is on this exact size
TARGET_HEIGHT, TARGET_WIDTH = 584, 565 

def read_image_any(path):
    # Try OpenCV first
    img = cv2.imread(path)
    if img is not None:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")  # Convert to 3-channel RGB
            return np.array(im)
    except Exception as e:
        print(f"Could not read image: {path} ({e})")
        return None

def load_image():
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        for file_name in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file_name)
            img = read_image_any(file_path)
            if img is None:
                continue

            # Resize all images to target_size
            img = cv2.resize(img, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)

            

            # Convert masks and labels to grayscale
            if folder_name == "eyes":
                images.append(img)
            elif folder_name == "labels":
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                labels.append(gray)
            elif folder_name == "masks":
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                masks.append(gray)

load_image()

In [3]:
# Augmentation pipeline from LAB2
train_transform = A.Compose([
    ### Geometric transforms
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.Affine(
            scale=[0.95, 1.05], # Zoom out by 5% or zoom in by 5%
            # Negative means it can be translated, rotated, sheered to different directions
            translate_percent=[-0.02, 0.02], 
            rotate=[-90, 90],
            shear=[-5, 5], 
            interpolation=cv2.INTER_LINEAR,
            mask_interpolation=cv2.INTER_NEAREST,
            rotate_method="ellipse",
            balanced_scale=True,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.7
        ),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.7
    ),
    A.HueSaturationValue(
        hue_shift_limit=3,
        sat_shift_limit=3,
        val_shift_limit=3,
        p=0.7
    ),
    A.CLAHE(
        clip_limit = 2,
        p=0.4
    ),
    A.OneOf([
        A.ElasticTransform(alpha=50, sigma=5, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        A.OpticalDistortion(distort_limit=0.05, p=0.3),
    ])
],
                            additional_targets={"mask": "mask", "label": "mask"})

In [4]:
# Augment the dataset, each image n time
def augment_dataset(images, masks, labels, transform, n):
    aug_images, aug_masks, aug_labels = [], [], []

    for i in range(len(images)):
        img, msk, lbl = images[i], masks[i], labels[i]

        # Always include original
        aug_images.append(img)
        aug_masks.append(msk)
        aug_labels.append(lbl)

        # Add n augmented versions
        for _ in range(n):
            augmented = transform(image=img, mask=msk, label=lbl)
            aug_images.append(augmented["image"])
            aug_masks.append(augmented["mask"])
            aug_labels.append(augmented["label"])

    return aug_images, aug_masks, aug_labels
aug_images, aug_masks, aug_labels = augment_dataset(images, masks, labels, transform=train_transform, n=4)

In [5]:
# Custom dataset class here, turn all the images into pytorch tensor and normalize it, and combine mask with eyes
class FundusDataset(Dataset):
    def __init__(self, images, masks, labels):
        self.images = images
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        label = self.labels[idx]

        # Convert numpy arrays to tensors if necessary
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image)
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        if isinstance(label, np.ndarray):
            label = torch.from_numpy(label)

        image = image.float() / 255.0
        mask = mask.float() / 255.0
        label = label.float() / 255.0
        
        if label.ndim == 2:
            label = label.unsqueeze(0)
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        if image.ndim == 3:
            image = image.permute(2, 0, 1)

        # Combine image and mask into one input
        x = torch.cat([image, mask], dim=0)
        
        # For some reason the height and width are swapped so I am enforcing them here
        H, W = 584, 565
        if x.shape[1:] != (H, W):
            x = torch.nn.functional.interpolate(
                x.unsqueeze(0), size=(H, W), mode="bilinear", align_corners=False
            ).squeeze(0)
        if label.shape[1:] != (H, W):
            label = torch.nn.functional.interpolate(
                label.unsqueeze(0), size=(H, W), mode="nearest"
            ).squeeze(0)
            
        return x, label  # model input and target

In [6]:
# train val split 80 20
dataset = FundusDataset(aug_images, aug_masks, aug_labels)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [7]:
# Data loaders with batch size 8
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [8]:
# model setup, in_channels = 4 (3 channels from eyes + 1 from mask) and num_classes = 1 (directly from labels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Pre trained U Net
model = Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=dataset[0][0].shape[0],
    classes=dataset[0][1].shape[0],      
).to(device)

In [9]:
# hyperparameters, epochs, loss function, optimizer
criterion = torch.nn.BCEWithLogitsLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
num_epochs = 10

In [10]:
# this is a helper function, since for some reason resizing is acting weird as I have handled before in the dataset class
# the inconviniet size of the smallest image kinda messes with the output shape
# neither height nor width is divisable by 16 so I am forcing my input and output to be the same shape here instead of crashing the training
def crop_to_output(label, output):
            _, _, H_out, W_out = output.shape
            return label[:, :, :H_out, :W_out]

In [11]:
# F1 - AUC - IOU Metrics
f1_train = F1Score(task='binary').to(device)
auc_train = AUROC(task='binary').to(device)
iou_train = JaccardIndex(task='binary').to(device)

f1_val = F1Score(task='binary').to(device)
auc_val = AUROC(task='binary').to(device)
iou_val = JaccardIndex(task='binary').to(device)

In [12]:
# Training Loop

best_val_f1 = 0.0
best_metrics = {}
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    f1_train.reset()
    auc_train.reset()
    iou_train.reset()
    
    train_loader_tqdm = tqdm(train_loader, desc=f'Epoch [{epoch+1}/{num_epochs}] - Training', leave=False, ascii=True)
    
    # Training
    for inputs, labels in train_loader_tqdm:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        # Ensure output and label shapes match
        labels_cropped = crop_to_output(labels, outputs)
        
        # BCEWithLogitsLoss expects raw logits, so no sigmoid here
        loss = criterion(outputs, labels_cropped)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        
        # Convert logits -> probs for metrics
        preds = torch.sigmoid(outputs)
        f1_train.update(preds, labels_cropped.int())
        auc_train.update(preds, labels_cropped.int())
        iou_train.update(preds, labels_cropped.int())

        train_loader_tqdm.set_postfix({'Train Loss': f'{loss.item():.4f}'})

    # Validation
    val_loss = 0.0
    model.eval()
    f1_val.reset()
    auc_val.reset()
    iou_val.reset()
    
    with torch.no_grad():
        val_loader_tqdm = tqdm(val_loader, desc=f'Epoch [{epoch+1}/{num_epochs}] - Validation', leave=False, ascii=True)
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            labels_cropped = crop_to_output(labels, outputs)
            
            loss = criterion(outputs, labels_cropped)
            val_loss += loss.item()

            preds = torch.sigmoid(outputs)
            f1_val.update(preds, labels_cropped.int())
            auc_val.update(preds, labels_cropped.int())
            iou_val.update(preds, labels_cropped.int())
            
            val_loader_tqdm.set_postfix({'Val Loss': f'{loss.item():.4f}'})

    # getting the metrics
    train_f1 = f1_train.compute()
    train_auc = auc_train.compute()
    train_iou = iou_train.compute()
    
    val_f1 = f1_val.compute()
    val_auc = auc_val.compute()
    val_iou = iou_val.compute()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | "
          f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | "
          f"Train IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_wts = copy.deepcopy(model.state_dict())
        best_metrics = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'train_f1': train_f1.item(),
            'val_f1': val_f1.item(),
            'train_auc': train_auc.item(),
            'val_auc': val_auc.item(),
            'train_iou': train_iou.item(),
            'val_iou': val_iou.item()
        }
        torch.save(best_model_wts, 'best_model.pth')
        print(f"Saved new best model at epoch {epoch+1} (Val F1: {val_f1:.4f})")
print("\nTraining Complete!")
print(f"Best Epoch: {best_metrics['epoch']}")
print(f"Best Validation F1: {best_metrics['val_f1']:.4f}")
print(f"Best Validation AUC: {best_metrics['val_auc']:.4f}")
print(f"Best Validation IoU: {best_metrics['val_iou']:.4f}")

Epoch [1/10] | Train Loss: 0.4900 | Val Loss: 0.3884 | Train F1: 0.1880 | Val F1: 0.3150 | Train AUC: 0.7172 | Val AUC: 0.7942 | Train IoU: 0.1038 | Val IoU: 0.1869
Saved new best model at epoch 1 (Val F1: 0.3150)


Epoch [2/10] | Train Loss: 0.3510 | Val Loss: 0.3033 | Train F1: 0.4673 | Val F1: 0.5590 | Train AUC: 0.8923 | Val AUC: 0.9187 | Train IoU: 0.3049 | Val IoU: 0.3880
Saved new best model at epoch 2 (Val F1: 0.5590)


Epoch [3/10] | Train Loss: 0.2894 | Val Loss: 0.2683 | Train F1: 0.5741 | Val F1: 0.6231 | Train AUC: 0.9335 | Val AUC: 0.9398 | Train IoU: 0.4026 | Val IoU: 0.4526
Saved new best model at epoch 3 (Val F1: 0.6231)


Epoch [4/10] | Train Loss: 0.2513 | Val Loss: 0.2439 | Train F1: 0.6167 | Val F1: 0.6514 | Train AUC: 0.9478 | Val AUC: 0.9525 | Train IoU: 0.4459 | Val IoU: 0.4831
Saved new best model at epoch 4 (Val F1: 0.6514)


Epoch [5/10] | Train Loss: 0.2267 | Val Loss: 0.2207 | Train F1: 0.6365 | Val F1: 0.6673 | Train AUC: 0.9550 | Val AUC: 0.9579 | Train IoU: 0.4668 | Val IoU: 0.5007
Saved new best model at epoch 5 (Val F1: 0.6673)


Epoch [6/10] | Train Loss: 0.2078 | Val Loss: 0.2024 | Train F1: 0.6502 | Val F1: 0.6761 | Train AUC: 0.9599 | Val AUC: 0.9599 | Train IoU: 0.4817 | Val IoU: 0.5106
Saved new best model at epoch 6 (Val F1: 0.6761)


Epoch [7/10] | Train Loss: 0.1937 | Val Loss: 0.1877 | Train F1: 0.6573 | Val F1: 0.6795 | Train AUC: 0.9626 | Val AUC: 0.9623 | Train IoU: 0.4895 | Val IoU: 0.5146
Saved new best model at epoch 7 (Val F1: 0.6795)


Epoch [8/10] | Train Loss: 0.1815 | Val Loss: 0.1744 | Train F1: 0.6633 | Val F1: 0.6921 | Train AUC: 0.9653 | Val AUC: 0.9642 | Train IoU: 0.4962 | Val IoU: 0.5292
Saved new best model at epoch 8 (Val F1: 0.6921)


Epoch [9/10] | Train Loss: 0.1722 | Val Loss: 0.1655 | Train F1: 0.6665 | Val F1: 0.6996 | Train AUC: 0.9665 | Val AUC: 0.9655 | Train IoU: 0.4998 | Val IoU: 0.5380
Saved new best model at epoch 9 (Val F1: 0.6996)


Epoch [10/10] | Train Loss: 0.1634 | Val Loss: 0.1590 | Train F1: 0.6746 | Val F1: 0.6951 | Train AUC: 0.9689 | Val AUC: 0.9675 | Train IoU: 0.5090 | Val IoU: 0.5326

Training Complete!
Best Epoch: 9
Best Validation F1: 0.6996
Best Validation AUC: 0.9655
Best Validation IoU: 0.5380


In [ ]:
test_model = model.load_state_dict(torch.load('best_model.pth'))

test_model = model.load_state_dict(torch.load('best_model-adamw.pth'))


In [14]:
# load testing. same data loader as before just changed it a little bit for testing
base_path = "test/"
images = []
masks = []
labels1 = []
labels2 = []


# have to resize all images to the same size, it does speed things up in the future but mostly because some of the data is on this exact size
TARGET_HEIGHT, TARGET_WIDTH = 584, 565 

def read_image_any(path):
    # Try OpenCV first
    img = cv2.imread(path)
    if img is not None:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")  # Convert to 3-channel RGB
            return np.array(im)
    except Exception as e:
        print(f"Could not read image: {path} ({e})")
        return None

def load_image():
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        for file_name in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file_name)
            img = read_image_any(file_path)
            if img is None:
                continue

            # Resize all images to target_size
            img = cv2.resize(img, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)

            

            # Convert masks and labels to grayscale
            if folder_name == "images":
                images.append(img)
            elif folder_name == "1st_manual":
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                labels1.append(gray)
            elif folder_name == "2nd_manual":
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                labels2.append(gray)
            elif folder_name == "mask":
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                masks.append(gray)

load_image()

In [15]:
l1 = FundusDataset(images, masks, labels1)
l2 = FundusDataset(images, masks, labels2)
test_loader1 = DataLoader(l1, batch_size=1, shuffle=False)
test_loader2 = DataLoader(l2, batch_size=1, shuffle=False)

In [16]:
# testing metrics. For some reason metrics from training didnt work so I had to change it a little bit
from torchmetrics.classification import BinaryF1Score, BinaryAUROC, BinaryJaccardIndex
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

f1_m1 = BinaryF1Score(threshold=0.5).to(device)
auc_m1 = BinaryAUROC().to(device)
iou_m1 = BinaryJaccardIndex(threshold=0.5).to(device)

f1_m2 = BinaryF1Score(threshold=0.5).to(device)
auc_m2 = BinaryAUROC().to(device)
iou_m2 = BinaryJaccardIndex(threshold=0.5).to(device)

f1_union = BinaryF1Score(threshold=0.5).to(device)
auc_union = BinaryAUROC().to(device)
iou_union = BinaryJaccardIndex(threshold=0.5).to(device)

In [17]:
# load the best model
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [19]:
# testing loop
with torch.no_grad():
    for idx in tqdm(range(len(l1)), desc="Testing"):
        # get data from both datasets
        x1, y1 = l1[idx]
        x2, y2 = l2[idx]   # should correspond to same image

        # forward pass
        x = x1.unsqueeze(0).to(device)   # [1, C, H, W]
        out = model(x)                   # logits [1, 1, H_out, W_out]
        prob = torch.sigmoid(out)        # probabilities [1,1,H_out,W_out]

        # prepare ground truths
        y1_b = y1.unsqueeze(0).to(device)
        y2_b = y2.unsqueeze(0).to(device)
        if y1_b.dim() == 3:
            y1_b = y1_b.unsqueeze(1)
        if y2_b.dim() == 3:
            y2_b = y2_b.unsqueeze(1)

        # crop to model output size
        y1_c = crop_to_output(y1_b, out)
        y2_c = crop_to_output(y2_b, out)

        # binarize & make sure dtype/device consistent
        y1_bin = (y1_c > 0.5).int()
        y2_bin = (y2_c > 0.5).int()
        y_union = ((y1_bin > 0) | (y2_bin > 0)).int()
        prob_b = prob.detach().float()

        # --- update metrics ---
        f1_m1.update(prob_b, y1_bin)
        auc_m1.update(prob_b, y1_bin)
        iou_m1.update(prob_b, y1_bin)

        f1_m2.update(prob_b, y2_bin)
        auc_m2.update(prob_b, y2_bin)
        iou_m2.update(prob_b, y2_bin)

        f1_union.update(prob_b, y_union)
        auc_union.update(prob_b, y_union)
        iou_union.update(prob_b, y_union)

        # --- save predicted mask ---
        pred_thresh = (prob_b >= 0.5).cpu().squeeze().numpy().astype(np.uint8) * 255
        Image.fromarray(pred_thresh).save(os.path.join("preds-test", f"pred_{idx:03d}.png"))


Testing: 100%|██████████| 20/20 [00:01<00:00, 13.36it/s]


In [20]:
m1_f1, m1_auc, m1_iou = f1_m1.compute().item(), auc_m1.compute().item(), iou_m1.compute().item()
m2_f1, m2_auc, m2_iou = f1_m2.compute().item(), auc_m2.compute().item(), iou_m2.compute().item()
u_f1, u_auc, u_iou = f1_union.compute().item(), auc_union.compute().item(), iou_union.compute().item()
print("\nTest Summary (averages across all test images):")
print(f"MANUAL1 -> F1: {m1_f1:.4f} | AUC: {m1_auc:.4f} | IoU: {m1_iou:.4f}")
print(f"MANUAL2 -> F1: {m2_f1:.4f} | AUC: {m2_auc:.4f} | IoU: {m2_iou:.4f}")
print(f"UNION   -> F1: {u_f1:.4f} | AUC: {u_auc:.4f} | IoU: {u_iou:.4f}")


Test Summary (averages across all test images):
MANUAL1 -> F1: 0.7671 | AUC: 0.9697 | IoU: 0.6222
MANUAL2 -> F1: 0.7873 | AUC: 0.9745 | IoU: 0.6493
UNION   -> F1: 0.7546 | AUC: 0.9659 | IoU: 0.6059
